# 08 — Unified Evaluation v2 (YOLOv8 v2 + Faster R-CNN)

Models evaluated:
- **YOLOv8 v1**: trained at 640px
- **YOLOv8 v2**: trained at 1280px + copy-paste augmentation
- **Faster R-CNN v1**: trained at 800px, no augmentation
- **Faster R-CNN v2**: trained at 1024px + RFS + augmentation

In [ ]:
from pathlib import Path

_here = Path.cwd().resolve()
for _root in (_here, *_here.parents):
    if (_root / "src").is_dir() and (_root / "outputs").is_dir():
        break
else:
    raise FileNotFoundError("Repo root not found (need src/ and outputs/).")

print(f"Repo root: {_root}")

In [ ]:
!pip install ultralytics --no-deps

In [ ]:
import sys
from pathlib import Path

_here = Path.cwd().resolve()
for _root in (_here, *_here.parents):
    if (_root / "src").is_dir() and (_root / "outputs").is_dir():
        break
else:
    raise FileNotFoundError("Repo root not found (need src/ and outputs/).")

if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import os
import json
import torch
import numpy as np
import pandas as pd
import cv2
import random
import importlib
import src.evaluate
importlib.reload(src.evaluate)

from src.utils import SEED, CLASS_MAP, FRCNN_CLASS_MAP, NUM_CLASSES, CLASS_NAMES, seed_everything
from src.fasterrcnn_dataset import BDD100KDataset
from src.fasterrcnn_utils import build_fasterrcnn, collate_fn
from src.evaluate import (
    predict_yolov8, predict_fasterrcnn,
    compute_map, compute_precision_recall_f1,
    measure_fps, get_model_size_mb, count_all_params,
    plot_pr_curve, build_confusion_matrix,
    draw_boxes, plot_sample_detections, plot_per_class_ap,
    build_pr_curve_data, filter_predictions_by_score,
)

seed_everything(SEED)
print(f"Repo root: {_root}")

## 1. Load Models

Load both v1 (baseline) and v2 (improved) models for comparison.

In [ ]:
from ultralytics import YOLO

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

YOLO_V1_PATH = _root / "outputs" / "bdd100k_project" / "runs" / "trained" / "yolov8m_bdd100k_best.pt"
YOLO_V2_PATH = _root / "outputs" / "bdd100k_project" / "runs" / "yolov8m_bdd100k_v2" / "weights" / "best.pt"

FRCNN_V1_PATH = _root / "outputs" / "bdd100k_project" / "fasterrcnn_best.pth"
FRCNN_V2_PATH = _root / "outputs" / "bdd100k_project" / "fasterrcnn_v2_best.pth"

yolo_v1 = YOLO(str(YOLO_V1_PATH))
yolo_v2 = YOLO(str(YOLO_V2_PATH)) if YOLO_V2_PATH.exists() else None

frcnn_v1 = build_fasterrcnn(num_classes=NUM_CLASSES)
frcnn_v1.load_state_dict(torch.load(FRCNN_V1_PATH, map_location=device))
frcnn_v1.to(device).eval()

frcnn_v2 = None
if FRCNN_V2_PATH.exists():
    from src.fasterrcnn_utils import build_fasterrcnn as build_frcnn_v2
    frcnn_v2_model = build_fasterrcnn(num_classes=NUM_CLASSES)
    frcnn_v2_model.load_state_dict(torch.load(FRCNN_V2_PATH, map_location=device))
    frcnn_v2_model.to(device).eval()

print(f"YOLOv8 v1: {YOLO_V1_PATH}")
print(f"YOLOv8 v2: {YOLO_V2_PATH} (exists={YOLO_V2_PATH.exists()})")
print(f"Faster R-CNN v1: {FRCNN_V1_PATH}")
print(f"Faster R-CNN v2: {FRCNN_V2_PATH} (exists={FRCNN_V2_PATH.exists()})")

## 2. Load Test Dataset

In [ ]:
DATASET_ROOT = _root / "outputs" / "bdd100k_preprocessing"

test_dataset = BDD100KDataset(
    image_dir=str(DATASET_ROOT / "bdd100k-yolo-subset-v1" / "images" / "test"),
    annotation_file=str(DATASET_ROOT / "test_annotations.json"),
    class_map=FRCNN_CLASS_MAP,
)

print(f"Test samples: {len(test_dataset)}")

## 3. Run Inference on Test Set

In [ ]:
from tqdm import tqdm
from torch.utils.data import DataLoader

test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, collate_fn=collate_fn)

yolo_v1_predictions = []
yolo_v2_predictions = []
frcnn_v1_predictions = []
frcnn_v2_predictions = []
all_targets = []

for images, targets in tqdm(test_loader, desc="Running inference"):
    img_tensor = images[0]
    target = targets[0]

    image_id = target["image_id"].item()
    item = test_dataset.annotations[image_id]

    image_name = item["name"]
    if not os.path.splitext(image_name)[1]:
        image_name = f"{image_name}.jpg"

    img_path = os.path.join(test_dataset.image_dir, image_name)

    yolo_v1_preds = predict_yolov8(yolo_v1, img_path)
    frcnn_v1_preds = predict_fasterrcnn(frcnn_v1, img_tensor, device)

    yolo_v2_preds = predict_yolov8(yolo_v2, img_path) if yolo_v2 else []
    frcnn_v2_preds = predict_fasterrcnn(frcnn_v2_model, img_tensor, device) if frcnn_v2_model else []

    def to_prediction_dict(preds):
        return {
            "boxes": torch.tensor([p["box"] for p in preds], dtype=torch.float32) if preds else torch.zeros((0, 4)),
            "scores": torch.tensor([p["score"] for p in preds]) if preds else torch.zeros(0),
            "labels": torch.tensor([p["label"] for p in preds], dtype=torch.int64) if preds else torch.zeros(0, dtype=torch.int64),
        }

    yolo_v1_predictions.append(to_prediction_dict(yolo_v1_preds))
    yolo_v2_predictions.append(to_prediction_dict(yolo_v2_preds))
    frcnn_v1_predictions.append(to_prediction_dict(frcnn_v1_preds))
    frcnn_v2_predictions.append(to_prediction_dict(frcnn_v2_preds))

    all_targets.append({
        "boxes": target["boxes"],
        "labels": target["labels"]
    })

print(f"Inference complete: {len(all_targets)} images")

## 4. mAP Computation

In [ ]:
!{sys.executable} -m pip install faster-coco-eval

In [ ]:
print("Computing mAP for YOLOv8 v1...")
yolo_v1_map = compute_map(yolo_v1_predictions, all_targets)

print("Computing mAP for YOLOv8 v2...")
yolo_v2_map = compute_map(yolo_v2_predictions, all_targets) if yolo_v2 else None

print("Computing mAP for Faster R-CNN v1...")
frcnn_v1_map = compute_map(frcnn_v1_predictions, all_targets)

print("Computing mAP for Faster R-CNN v2...")
frcnn_v2_map = compute_map(frcnn_v2_predictions, all_targets) if frcnn_v2_model else None

def print_map_results(name, result):
    if result is None:
        print(f"\n--- {name} ---")
        print("Not available (model not trained)")
        return
    print(f"\n--- {name} ---")
    print(f"mAP@0.5:0.95: {result['map'].item():.4f}")
    print(f"mAP@0.5:      {result['map_50'].item():.4f}")
    print(f"mAP@0.75:     {result['map_75'].item():.4f}")
    print(f"mAP (small):  {result['map_small'].item():.4f}")
    print(f"mAP (medium): {result['map_medium'].item():.4f}")
    print(f"mAP (large):  {result['map_large'].item():.4f}")

print_map_results("YOLOv8 v1 (640px baseline)", yolo_v1_map)
print_map_results("YOLOv8 v2 (1280px + copy-paste)", yolo_v2_map)
print_map_results("Faster R-CNN v1 (800px baseline)", frcnn_v1_map)
print_map_results("Faster R-CNN v2 (1024px + RFS + aug)", frcnn_v2_map)

## 5. Summary Table

In [ ]:
summary_data = [
    {"Model": "YOLOv8 v1 (640px)", "mAP@0.5:0.95": yolo_v1_map["map"].item(), 
     "mAP@0.5": yolo_v1_map["map_50"].item(), "mAP@0.75": yolo_v1_map["map_75"].item(),
     "mAP small": yolo_v1_map["map_small"].item(), "mAP medium": yolo_v1_map["map_medium"].item()},
]

if yolo_v2_map:
    summary_data.append({"Model": "YOLOv8 v2 (1280px)", "mAP@0.5:0.95": yolo_v2_map["map"].item(),
         "mAP@0.5": yolo_v2_map["map_50"].item(), "mAP@0.75": yolo_v2_map["map_75"].item(),
         "mAP small": yolo_v2_map["map_small"].item(), "mAP medium": yolo_v2_map["map_medium"].item()})

summary_data.append({"Model": "Faster R-CNN v1 (800px)", "mAP@0.5:0.95": frcnn_v1_map["map"].item(),
     "mAP@0.5": frcnn_v1_map["map_50"].item(), "mAP@0.75": frcnn_v1_map["map_75"].item(),
     "mAP small": frcnn_v1_map["map_small"].item(), "mAP medium": frcnn_v1_map["map_medium"].item()})

if frcnn_v2_map:
    summary_data.append({"Model": "Faster R-CNN v2 (1024px)", "mAP@0.5:0.95": frcnn_v2_map["map"].item(),
         "mAP@0.5": frcnn_v2_map["map_50"].item(), "mAP@0.75": frcnn_v2_map["map_75"].item(),
         "mAP small": frcnn_v2_map["map_small"].item(), "mAP medium": frcnn_v2_map["map_medium"].item()})

summary_df = pd.DataFrame(summary_data)
summary_df

## 6. Per-Class AP Comparison

In [ ]:
from IPython.display import Image, display

CLASS_NAMES_LIST = [CLASS_NAMES[i] for i in range(len(CLASS_MAP))]

yolo_v1_per_class = yolo_v1_map["map_per_class"].numpy()
frcnn_v1_per_class = frcnn_v1_map["map_per_class"].numpy()

per_class_data = {"Class": CLASS_NAMES_LIST}
per_class_data["YOLOv8 v1"] = yolo_v1_per_class
per_class_data["F-RCNN v1"] = frcnn_v1_per_class

if yolo_v2_map is not None:
    per_class_data["YOLOv8 v2"] = yolo_v2_map["map_per_class"].numpy()
    per_class_data["YOLOv8 Δ"] = yolo_v2_map["map_per_class"].numpy() - yolo_v1_per_class

if frcnn_v2_map is not None:
    per_class_data["F-RCNN v2"] = frcnn_v2_map["map_per_class"].numpy()
    per_class_data["F-RCNN Δ"] = frcnn_v2_map["map_per_class"].numpy() - frcnn_v1_per_class

per_class_df = pd.DataFrame(per_class_data)
print(per_class_df.to_string(index=False))

per_class_plot_path = _root / "outputs" / "bdd100k_project" / "results" / "plots" / "per_class_ap_comparison_v2.png"

if yolo_v2_map is not None and frcnn_v2_map is not None:
    plot_per_class_ap(
        np.concatenate([yolo_v1_per_class, yolo_v2_map["map_per_class"].numpy()]),
        np.concatenate([frcnn_v1_per_class, frcnn_v2_map["map_per_class"].numpy()]),
        [f"{n}_v1" for n in CLASS_NAMES_LIST] + [f"{n}_v2" for n in CLASS_NAMES_LIST],
        save_path=str(per_class_plot_path)
    )
    display(Image(filename=str(per_class_plot_path)))

## 7. Before vs After Comparison

Generate side-by-side comparison tables and plots.

In [ ]:
import matplotlib.pyplot as plt

classes = ["car", "person", "truck", "bus", "motor", "bike", "tl", "ts"]

if yolo_v2_map is not None and frcnn_v2_map is not None:
    yolo_v1 = yolo_v1_map["map_per_class"].numpy()
    yolo_v2 = yolo_v2_map["map_per_class"].numpy()
    frcnn_v1 = frcnn_v1_map["map_per_class"].numpy()
    frcnn_v2 = frcnn_v2_map["map_per_class"].numpy()

    x = np.arange(len(classes))
    w = 0.2

    fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)

    axes[0].bar(x - w/2, yolo_v1, w, label="v1 (640px)", color="#9BBFE0", alpha=0.9)
    axes[0].bar(x + w/2, yolo_v2, w, label="v2 (1280px)", color="#2A6DB5", alpha=0.9)
    axes[0].set_xticks(x); axes[0].set_xticklabels(classes, rotation=15)
    axes[0].set_ylabel("AP @ IoU 0.5:0.95"); axes[0].set_ylim(0, 0.55)
    axes[0].set_title("YOLOv8 — v1 vs v2"); axes[0].legend()
    axes[0].axhline(0.2, color="red", linestyle="--", alpha=0.4)

    axes[1].bar(x - w/2, frcnn_v1, w, label="v1 (800px)", color="#9BBFE0", alpha=0.9)
    axes[1].bar(x + w/2, frcnn_v2, w, label="v2 (1024px)", color="#2A6DB5", alpha=0.9)
    axes[1].set_xticks(x); axes[1].set_xticklabels(classes, rotation=15)
    axes[1].set_ylabel("AP @ IoU 0.5:0.95"); axes[1].set_ylim(0, 0.55)
    axes[1].set_title("Faster R-CNN — v1 vs v2"); axes[1].legend()
    axes[1].axhline(0.2, color="red", linestyle="--", alpha=0.4)

    plt.tight_layout()
    
    comparison_plot_path = _root / "outputs" / "bdd100k_project" / "results" / "plots" / "before_after_perclass.png"
    os.makedirs(os.path.dirname(comparison_plot_path), exist_ok=True)
    plt.savefig(str(comparison_plot_path), dpi=150)
    plt.show()

    print(f"Comparison plot saved to {comparison_plot_path}")

## 8. Save Comparison Results

In [ ]:
os.makedirs(_root / "outputs" / "bdd100k_project" / "results" / "metrics", exist_ok=True)

comparison_csv_path = _root / "outputs" / "bdd100k_project" / "results" / "metrics" / "comparison_before_after.csv"

comparison_results = []

for i, cls in enumerate(CLASS_NAMES_LIST):
    row = {"Class": cls}
    row["YOLO v1"] = yolo_v1_map["map_per_class"].numpy()[i]
    row["F-RCNN v1"] = frcnn_v1_map["map_per_class"].numpy()[i]
    if yolo_v2_map is not None:
        row["YOLO v2"] = yolo_v2_map["map_per_class"].numpy()[i]
        row["YOLO Δ"] = row["YOLO v2"] - row["YOLO v1"]
    if frcnn_v2_map is not None:
        row["F-RCNN v2"] = frcnn_v2_map["map_per_class"].numpy()[i]
        row["F-RCNN Δ"] = row["F-RCNN v2"] - row["F-RCNN v1"]
    comparison_results.append(row)

comparison_df = pd.DataFrame(comparison_results)
comparison_df.to_csv(comparison_csv_path, index=False)

print(f"Comparison CSV saved to {comparison_csv_path}")
print("\nComparison Results:")
print(comparison_df.to_string(index=False))